# PyTTI — Text-to-Image Generation in Google Colab

This notebook sets up [pytti-core](https://github.com/pytti-tools/pytti-core) for text-guided image generation on Google Colab.

**Requirements:** A GPU runtime (Runtime → Change runtime type → GPU).

## Sections
1. **Clone & Install** — clone repo, install all dependencies
2. **GPU Verification** — confirm CUDA is available and check VRAM
3. **Gradio UI Mode** — launch the full interactive UI with a public link
4. **Programmatic Mode** — render directly via pytti's Hydra config
5. **View & Download Outputs** — display and download rendered images

---
## 1. Clone & Install Dependencies

This cell clones the pytti-core repo (with vendor submodules), installs PyTorch for Colab's CUDA toolkit, then installs all pip and git-based dependencies.

**Run this cell once.** It takes a few minutes on first run.

In [ ]:
import subprocess, sys, os

REPO_DIR = "/content/pytti-core"

# ── Clone pytti-core with vendor submodules ──────────────────────────────────
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules -j8 https://github.com/pytti-tools/pytti-core.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

# ── Install PyTorch (use Colab's pre-installed CUDA) ─────────────────────────
# Colab provides CUDA 12.x — install a compatible PyTorch build.
# We override the pinned torch==1.13.1 from requirements.txt since Colab's
# CUDA toolkit is newer. PyTorch 2.x works with Colab and is backward-compatible
# with the pytti codebase.
!pip install -q torch torchvision torchaudio

# ── Install pip dependencies ─────────────────────────────────────────────────
# Install requirements.txt but skip the torch/torchvision lines (already installed)
# and relax strict pins that conflict with Colab's environment.
!pip install -q \
    numpy \
    Pillow \
    imageio \
    imageio-ffmpeg \
    omegaconf \
    hydra-core \
    pytorch-lightning \
    kornia \
    einops \
    transformers \
    ftfy \
    regex \
    tqdm \
    loguru \
    PyGLM \
    adjustText \
    exrex \
    matplotlib-label-lines \
    pandas \
    seaborn \
    scikit-learn \
    gdown \
    gradio

# ── Install vendor submodule packages ────────────────────────────────────────
!pip install -q {REPO_DIR}/vendor/AdaBins
!pip install -q {REPO_DIR}/vendor/CLIP
!pip install -q {REPO_DIR}/vendor/GMA
!pip install -q {REPO_DIR}/vendor/taming-transformers

# ── Install pytti-core itself ────────────────────────────────────────────────
!pip install -q {REPO_DIR}

# ── Download AdaBins checkpoint ──────────────────────────────────────────────
ADABINS_DIR = os.path.join(os.path.expanduser("~"), ".cache", "adabins")
ADABINS_PT = os.path.join(ADABINS_DIR, "AdaBins_nyu.pt")
if not os.path.exists(ADABINS_PT):
    os.makedirs(ADABINS_DIR, exist_ok=True)
    !wget -q -O {ADABINS_PT} https://github.com/hithereai/deforum-for-automatic1111-webui/releases/download/AdaBins/AdaBins_nyu.pt
    print(f"Downloaded AdaBins checkpoint to {ADABINS_PT}")
else:
    print("AdaBins checkpoint already exists.")

print("\n✅ Installation complete!")

### Apply gradio_client compatibility patches

Recent versions of `gradio` changed internal APIs that pytti relies on. This cell applies monkey-patches so the Gradio UI works correctly.

In [ ]:
# ── gradio_client monkey-patches ─────────────────────────────────────────────
# Newer gradio versions moved/renamed serialization utilities.
# Patch them so pytti's Gradio blocks launch without ImportError.
try:
    import gradio_client
    import gradio_client.utils

    # Patch: some pytti code expects gradio_client.serializing
    if not hasattr(gradio_client, "serializing"):
        import types
        gradio_client.serializing = types.ModuleType("gradio_client.serializing")
        sys.modules["gradio_client.serializing"] = gradio_client.serializing

    # Patch: ensure Serializable base class exists
    if not hasattr(gradio_client.serializing, "Serializable"):
        class Serializable:
            def serialize(self, x, *args, **kwargs):
                return x
            def deserialize(self, x, *args, **kwargs):
                return x
        gradio_client.serializing.Serializable = Serializable

    print("✅ gradio_client patches applied.")
except Exception as e:
    print(f"⚠️  Patch warning (non-fatal): {e}")

---
## 2. GPU Verification

Confirm that CUDA is available and check how much VRAM the runtime has. PyTTI needs a GPU — if this cell shows `CUDA available: False`, go to **Runtime → Change runtime type → GPU**.

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024 ** 3)
    print(f"Total VRAM      : {vram_gb:.1f} GB")
    if vram_gb < 8:
        print("⚠️  Less than 8 GB VRAM — you may need to reduce image resolution.")
    else:
        print("✅ VRAM looks good for default settings.")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → GPU.")

---
## 3. Rendering Mode A — Gradio Interactive UI

Launch pytti's full Gradio interface with a **public share link** you can open in any browser. This gives you sliders and text fields for all parameters.

After running this cell, click the **public URL** printed in the output to open the UI.

In [ ]:
import subprocess, os, signal, sys

os.chdir("/content/pytti-core")

# Try to find and launch the Gradio app.
# pytti-core may include a gradio app script — look for common names.
gradio_candidates = ["app.py", "gradio_app.py", "ui.py", "webui.py"]
gradio_script = None
for name in gradio_candidates:
    if os.path.exists(name):
        gradio_script = name
        break

if gradio_script:
    print(f"Launching Gradio UI from {gradio_script}...")
    !python {gradio_script}
else:
    # Fall back: build a minimal Gradio wrapper around pytti's workhorse
    print("No built-in Gradio app found. Launching a minimal Gradio wrapper...")

    import gradio as gr
    from omegaconf import OmegaConf

    sys.path.insert(0, "/content/pytti-core/src")
    from pytti.workhorse import _main as pytti_main

    def render(prompt, steps, width, height, image_model, seed):
        """Run a pytti render and return the final image."""
        from hydra import compose, initialize_config_dir
        from hydra.core.global_hydra import GlobalHydra
        from pathlib import Path
        from PIL import Image
        import glob

        GlobalHydra.instance().clear()
        config_dir = os.path.join("/content/pytti-core/src/pytti/config")

        with initialize_config_dir(config_dir=config_dir, version_base=None):
            cfg = compose(config_name="default")

        cfg.scenes = prompt
        cfg.steps_per_scene = int(steps)
        cfg.width = int(width)
        cfg.height = int(height)
        cfg.image_model = image_model
        cfg.file_namespace = "gradio_run"
        if seed:
            cfg.seed = str(seed)

        pytti_main(cfg)

        # Find the most recent output image
        out_dir = "/content/pytti-core/images_out/gradio_run/"
        images = sorted(glob.glob(os.path.join(out_dir, "*.png")), key=os.path.getmtime)
        if images:
            return Image.open(images[-1])
        return None

    demo = gr.Interface(
        fn=render,
        inputs=[
            gr.Textbox(label="Prompt", value="a beautiful sunset over the ocean", lines=3),
            gr.Slider(10, 500, value=100, step=10, label="Steps per scene"),
            gr.Slider(64, 512, value=180, step=4, label="Width"),
            gr.Slider(64, 512, value=112, step=4, label="Height"),
            gr.Dropdown(
                ["Unlimited Palette", "Limited Palette", "VQGAN"],
                value="Unlimited Palette",
                label="Image Model"
            ),
            gr.Textbox(label="Seed (optional)", value=""),
        ],
        outputs=gr.Image(label="Rendered Image"),
        title="PyTTI — Text-to-Image",
        description="Generate images from text prompts using CLIP-guided optimization.",
    )

    demo.launch(share=True)

---
## 4. Rendering Mode B — Programmatic Rendering

Call pytti's rendering engine directly with custom parameters. Edit the config below and run the cell.

This is useful for batch rendering, scripting, or when you want fine-grained control over every parameter.

In [ ]:
import os, sys, glob
from pathlib import Path

os.chdir("/content/pytti-core")
sys.path.insert(0, "/content/pytti-core/src")

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

# ── Configure your render ────────────────────────────────────────────────────
# Edit these parameters to customize the output.

PROMPT       = "a sprawling cyberpunk city at night, neon lights reflecting on wet streets"
STEPS        = 150       # More steps = more detail (and more time)
WIDTH        = 180       # Image width in pixels (before pixel_size multiplier)
HEIGHT       = 112       # Image height in pixels
IMAGE_MODEL  = "Unlimited Palette"  # Options: "Unlimited Palette", "Limited Palette", "VQGAN"
NAMESPACE    = "colab_render"       # Output folder name under images_out/
SAVE_EVERY   = 50        # Save a snapshot every N steps

# ── Load default config and apply overrides ──────────────────────────────────
GlobalHydra.instance().clear()
config_dir = os.path.abspath("src/pytti/config")

with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name="default")

cfg.scenes = PROMPT
cfg.steps_per_scene = STEPS
cfg.width = WIDTH
cfg.height = HEIGHT
cfg.image_model = IMAGE_MODEL
cfg.file_namespace = NAMESPACE
cfg.save_every = SAVE_EVERY
cfg.display_every = SAVE_EVERY

print("Render config:")
print(f"  Prompt      : {cfg.scenes}")
print(f"  Steps       : {cfg.steps_per_scene}")
print(f"  Size        : {cfg.width}x{cfg.height}")
print(f"  Model       : {cfg.image_model}")
print(f"  Output dir  : images_out/{cfg.file_namespace}/")
print()

# ── Run the render ───────────────────────────────────────────────────────────
from pytti.workhorse import _main as pytti_main
pytti_main(cfg)

print("\n✅ Render complete!")

---
## 5. View & Download Rendered Outputs

Display the images from your most recent render and provide download links.

Change `OUTPUT_NAMESPACE` below if you used a different `file_namespace` in the render cell.

In [ ]:
import glob, os
from IPython.display import display, Image as IPImage, HTML
from google.colab import files as colab_files

OUTPUT_NAMESPACE = "colab_render"  # Must match the NAMESPACE used during rendering
OUTPUT_DIR = f"/content/pytti-core/images_out/{OUTPUT_NAMESPACE}/"

image_paths = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")), key=os.path.getmtime)

if not image_paths:
    print(f"No images found in {OUTPUT_DIR}")
    print("Run a render cell first (Section 3 or 4).")
else:
    print(f"Found {len(image_paths)} image(s) in {OUTPUT_DIR}\n")

    # Show the final image large
    print("=" * 50)
    print("FINAL OUTPUT")
    print("=" * 50)
    display(IPImage(filename=image_paths[-1], width=512))

    # Show progression if multiple snapshots exist
    if len(image_paths) > 1:
        print(f"\nProgression ({len(image_paths)} snapshots):")
        print("-" * 50)
        for path in image_paths:
            print(os.path.basename(path))
            display(IPImage(filename=path, width=256))

    # Download button for the final image
    print("\n📥 Downloading final image...")
    colab_files.download(image_paths[-1])

### Download all outputs as a zip

Use this cell to download every image from the render as a single zip file.

In [ ]:
import zipfile, os, glob
from google.colab import files as colab_files

OUTPUT_NAMESPACE = "colab_render"
OUTPUT_DIR = f"/content/pytti-core/images_out/{OUTPUT_NAMESPACE}/"
ZIP_PATH = f"/content/{OUTPUT_NAMESPACE}_outputs.zip"

image_paths = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))

if not image_paths:
    print(f"No images found in {OUTPUT_DIR}")
else:
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in image_paths:
            zf.write(path, os.path.basename(path))
    print(f"Created {ZIP_PATH} with {len(image_paths)} image(s).")
    colab_files.download(ZIP_PATH)